# ScienceQA Visual Challenge: Starter Notebook

This notebook provides a starting point for the ScienceQA Visual Multiple-Choice Challenge. It is based on the provided baseline solution, but has been adapted to be more of a general-purpose starter.

**Objective:** Build a model that can answer visual multiple-choice questions based on scientific diagrams and text.

**Baseline Model:** `HuggingFaceTB/SmolVLM-500M-Instruct` (~500 M params)
**Fine-Tuning:** QLoRA (4-bit NF4)
**Scoring:** Multiple-choice log-likelihood

---

In [1]:
# ── 0. Install libraries ──────────────────────────────────────────
# Run this cell to install the necessary Python packages.
%pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 146.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.1 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ── 1. Imports & Configuration ───────────────────────────────────────────────
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Paths ────────────────────────────────────────────────────────────────────
# Adjust these paths to match your local environment
DATA_DIR = Path("/content/drive/MyDrive/final_dl")

# ── Model ────────────────────────────────────────────────────────────────────
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# ── Basic Settings ───────────────────────────────────────────────────────────
# Higher resolution helps ScienceQA diagrams, maps, axes, and embedded text.
IMG_SIZE = 384

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA A100-SXM4-40GB


## 2. Load and Preprocess Data

In [4]:
# ── 2a. Load CSVs ─────────────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

# The 'choices' column is a JSON string, so we parse it
for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
train_df.head(2)

Train: 3,109 | Val: 1,048 | Test: 1,008


,id,image_path,question,choices,num_choices,answer,hint,lecture,solution,task,grade,subject,topic,category,skill
0,train_07667,images/train/train_07667.png,Why might putting each tadpole in its own pool...,[the male's tadpoles will be larger when they ...,3,2,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...
1,train_02628,images/train/train_02628.png,Why might forming strong social bonds with oth...,"[the female's offspring will live longer, the ...",3,0,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...


In [5]:
print("train:", len(train_df))
print("val:", len(val_df))
print("test:", len(test_df))

train: 3109
val: 1048
test: 1008


In [6]:
from transformers import AutoProcessor, AutoModelForVision2Seq

CHOICE_LETTERS = "ABCDEFGHIJ"
USE_METADATA_IN_PROMPT = True
PROMPT_METADATA_COLUMNS = ["subject", "grade", "topic"]
ANSWER_PREFIX = "The correct answer is:"


def format_prompt_metadata(row: pd.Series) -> str:
    if not USE_METADATA_IN_PROMPT:
        return ""

    label_map = {
        "subject": "Subject",
        "grade": "Grade",
        "topic": "Topic",
    }
    metadata_lines = []

    for col in PROMPT_METADATA_COLUMNS:
        value = row.get(col, "")
        if pd.notna(value) and str(value).strip():
            label = label_map.get(col, col.replace("_", " ").title())
            metadata_lines.append(f"{label}: {str(value).strip()}")

    return "\n".join(metadata_lines)


def build_prompt(row: pd.Series, include_answer: bool = False) -> str:
    context_parts = []
    lecture = row.get("lecture", "")
    hint = row.get("hint", "")

    if pd.notna(lecture) and str(lecture).strip():
        context_parts.append(str(lecture).strip())
    if pd.notna(hint) and str(hint).strip():
        context_parts.append(str(hint).strip())

    metadata_str = format_prompt_metadata(row)
    context_str = "\n".join(context_parts)

    choices = row["choices"]
    choices_str = "\n".join(
        f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )

    prompt = "<image>\n"
    prompt += "Use the image and the provided information to answer the multiple-choice question.\n\n"
    if metadata_str:
        prompt += f"Metadata:\n{metadata_str}\n\n"
    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n\n"
    prompt += "Reply with only one letter and nothing else.\n"
    prompt += ANSWER_PREFIX

    if include_answer:
        answer_idx = int(row["answer"])
        prompt += f" {CHOICE_LETTERS[answer_idx]}"

    return prompt

## 3. Model Training

In [7]:
%pip install -q transformers peft accelerate datasets

In [8]:
from transformers import AutoProcessor, AutoModelForVision2Seq
import torch

# 老师要求的模型
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# 设备
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

# 加载 processor
processor = AutoProcessor.from_pretrained(MODEL_ID)

# 有些模型没有 pad token，这里补一下
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

# 加载 model
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
)

# Gradient checkpointing 用计算换显存，方便 384x384 训练。
if hasattr(model.config, "use_cache"):
    model.config.use_cache = False
if hasattr(model, "gradient_checkpointing_enable"):
    try:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    except TypeError:
        model.gradient_checkpointing_enable()
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

# CPU 情况下手动放到 device
if not torch.cuda.is_available():
    model.to(device)

# 训练前保持 train mode；Trainer 会接管 train/eval 切换。
model.train()

print("Processor loaded.")
print("Model loaded.")
print("Gradient checkpointing enabled:", getattr(model, "is_gradient_checkpointing", "unknown"))
print("Using device:", model.device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Processor loaded.
Model loaded.
Gradient checkpointing enabled: True
Using device: cuda:0


In [9]:
from peft import LoraConfig, get_peft_model

USE_DORA = True

# Attention + MLP LoRA at lower rank. DoRA improves adapter quality by learning
# weight magnitude separately from direction, while staying within the 5M budget.
lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "v_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_dora=USE_DORA,
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"DoRA enabled: {USE_DORA}")
print(f"Trainable params budget check: {trainable_params:,} / 5,000,000")
if trainable_params > 5_000_000:
    raise ValueError("LoRA/DoRA trainable parameter count exceeds the 5M budget. Lower r or reduce target_modules.")

trainable params: 2,162,688 || all params: 509,644,992 || trainable%: 0.4244
DoRA enabled: True
Trainable params budget check: 2,162,688 / 5,000,000


In [10]:
# from torch.utils.data import Dataset
# from PIL import Image

# class VQATrainDataset(Dataset):
#     def __init__(self, df, processor, data_dir, img_size):
#         self.df = df.reset_index(drop=True)
#         self.processor = processor
#         self.data_dir = data_dir
#         self.img_size = img_size

#     def __len__(self):
#         return len(self.df)

#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]

#         image = Image.open(self.data_dir / row["image_path"]).convert("RGB").resize((self.img_size, self.img_size))
#         text = build_prompt(row, include_answer=True)

#         enc = self.processor(
#             text=[text],
#             images=[image],
#             return_tensors="pt",
#             padding=False,
#             truncation=False,
#         )

#         item = {}
#         for k, v in enc.items():
#             item[k] = v.squeeze(0)

#         item["labels"] = item["input_ids"].clone()
#         return item

from torch.utils.data import Dataset
from PIL import Image
import torch

class VQATrainDataset(Dataset):
    def __init__(self, df, processor, data_dir, img_size):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.data_dir = data_dir
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(self.data_dir / row["image_path"]).convert("RGB").resize((self.img_size, self.img_size))

        prefix_text = build_prompt(row, include_answer=False)
        full_text = build_prompt(row, include_answer=True)

        enc = self.processor(
            text=[full_text],
            images=[image],
            return_tensors="pt",
            padding=False,
            truncation=False,
        )

        prefix_enc = self.processor(
            text=[prefix_text],
            images=[image],
            return_tensors="pt",
            padding=False,
            truncation=False,
        )

        item = {}
        for k, v in enc.items():
            item[k] = v.squeeze(0)

        labels = item["input_ids"].clone()

        prefix_len = prefix_enc["input_ids"].shape[1]

        # 前面的 prompt 部分不参与 loss
        labels[:prefix_len] = -100

        item["labels"] = labels
        return item

In [11]:
import torch

def multimodal_collate_fn(batch):
    input_ids = [x["input_ids"] for x in batch]
    attention_mask = [x["attention_mask"] for x in batch]
    pixel_values = [x["pixel_values"] for x in batch]
    labels = [x["labels"] for x in batch]

    input_ids = torch.nn.utils.rnn.pad_sequence(
        input_ids,
        batch_first=True,
        padding_value=processor.tokenizer.pad_token_id,
    )

    attention_mask = torch.nn.utils.rnn.pad_sequence(
        attention_mask,
        batch_first=True,
        padding_value=0,
    )

    labels = torch.nn.utils.rnn.pad_sequence(
        labels,
        batch_first=True,
        padding_value=-100,
    )

    pixel_values = torch.stack(pixel_values, dim=0)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "pixel_values": pixel_values,
        "labels": labels,
    }

In [12]:
# train_small = train_df.sample(min(1000, len(train_df)), random_state=42).reset_index(drop=True)
# val_small = val_df.sample(min(200, len(val_df)), random_state=42).reset_index(drop=True)

train_full = train_df.reset_index(drop=True)
train_dataset = VQATrainDataset(train_full, processor, DATA_DIR, IMG_SIZE)


In [13]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./smolvlm_lora_out",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    learning_rate=2e-4,
    logging_steps=20,
    save_steps=200,
    eval_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=multimodal_collate_fn,
)

trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.


Step,Training Loss
20,0.729500
40,0.600500
60,0.480300
80,0.453700
100,0.631300
120,0.455700
140,0.435200
160,0.459300
180,0.337000
200,0.478100


TrainOutput(global_step=778, training_loss=0.36569086843407245, metrics={'train_runtime': 13208.5764, 'train_samples_per_second': 0.471, 'train_steps_per_second': 0.059, 'total_flos': 2.459503222549632e+16, 'train_loss': 0.36569086843407245, 'epoch': 2.0})

In [14]:
trainer.save_model("./smolvlm_lora_out/final")
processor.save_pretrained("./smolvlm_lora_out/final")
print("saved.")

saved.


In [15]:
from pathlib import Path

SAVE_DIR = Path("/content/drive/MyDrive/final_dl/my_trained_model")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(SAVE_DIR))
processor.save_pretrained(str(SAVE_DIR))

print("Saved to:", SAVE_DIR)

Saved to: /content/drive/MyDrive/final_dl/my_trained_model


In [16]:
from transformers import AutoProcessor, AutoModelForVision2Seq
from peft import PeftModel
import torch

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
SAVE_DIR = "/content/drive/MyDrive/final_dl/my_trained_model"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

processor = AutoProcessor.from_pretrained(SAVE_DIR)

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
)

model = PeftModel.from_pretrained(base_model, SAVE_DIR)

if not torch.cuda.is_available():
    model.to(device)

model.eval()

print("Loaded trained model from:", SAVE_DIR)
print("Using device:", model.device)

/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


Loaded trained model from: /content/drive/MyDrive/final_dl/my_trained_model
Using device: cuda:0


In [17]:
# 在 val 集上用 multiple-choice log-likelihood 验证

from tqdm.auto import tqdm
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F

CHOICE_LETTERS = "ABCDEFGHIJ"
ANALYSIS_COLUMNS = [
    "num_choices",
    "task",
    "grade",
    "subject",
    "topic",
    "category",
    "skill",
]

# 确保 batch 内不同候选答案按右侧 padding，便于用 prefix_len 截出答案 token。
processor.tokenizer.padding_side = "right"


def get_model_device(model):
    model_device = getattr(model, "device", None)
    if model_device is not None:
        return model_device
    return next(model.parameters()).device


def move_to_model_device(inputs, model):
    model_device = get_model_device(model)
    return {
        k: v.to(model_device) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }


def score_choices_by_loglikelihood(row, normalize_by_length=True):
    """Return one log-likelihood score per answer letter for a single VQA row."""
    image = Image.open(DATA_DIR / row["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    prefix_text = build_prompt(row, include_answer=False)
    num_choices = len(row["choices"])

    candidate_texts = [
        f"{prefix_text} {CHOICE_LETTERS[i]}"
        for i in range(num_choices)
    ]

    inputs = processor(
        text=candidate_texts,
        images=[image] * num_choices,
        return_tensors="pt",
        padding=True,
    )
    prefix_inputs = processor(
        text=[prefix_text],
        images=[image],
        return_tensors="pt",
        padding=False,
    )

    prefix_len = int(prefix_inputs["input_ids"].shape[1])
    inputs = move_to_model_device(inputs, model)
    input_ids = inputs["input_ids"]
    attention_mask = inputs.get("attention_mask")

    with torch.inference_mode():
        outputs = model(**inputs)

    log_probs = F.log_softmax(outputs.logits[:, :-1, :], dim=-1)
    target_ids = input_ids[:, 1:]
    token_log_probs = log_probs.gather(-1, target_ids.unsqueeze(-1)).squeeze(-1)

    scores = []
    token_counts = []
    start = max(prefix_len - 1, 0)

    for choice_idx in range(num_choices):
        if attention_mask is not None:
            seq_len = int(attention_mask[choice_idx].sum().item())
        else:
            seq_len = int(input_ids.shape[1])

        # shift 后的位置 j 预测原序列里的 token j+1；答案 token 从 prefix_len 开始。
        end = max(seq_len - 1, start + 1)
        answer_token_log_probs = token_log_probs[choice_idx, start:end]

        score = answer_token_log_probs.mean() if normalize_by_length else answer_token_log_probs.sum()
        scores.append(float(score.item()))
        token_counts.append(int(answer_token_log_probs.numel()))

    return scores, token_counts


def predict_row_by_loglikelihood(row, normalize_by_length=True):
    scores, token_counts = score_choices_by_loglikelihood(
        row,
        normalize_by_length=normalize_by_length,
    )
    pred_idx = int(np.argmax(scores))
    sorted_scores = sorted(scores, reverse=True)
    margin = float(sorted_scores[0] - sorted_scores[1]) if len(sorted_scores) > 1 else np.nan
    return pred_idx, scores, token_counts, margin


def evaluate_with_loglikelihood(df, desc="Evaluating with log-likelihood"):
    rows = []

    for i in tqdm(range(len(df)), desc=desc):
        row = df.iloc[i]
        pred_idx, scores, token_counts, margin = predict_row_by_loglikelihood(row)
        gt = int(row["answer"])

        result = {
            "id": row["id"],
            "pred": pred_idx,
            "gt": gt,
            "pred_letter": CHOICE_LETTERS[pred_idx],
            "gt_letter": CHOICE_LETTERS[gt],
            "correct": int(pred_idx == gt),
            "score_margin": margin,
            "question": row["question"],
        }

        for col in ANALYSIS_COLUMNS:
            if col in row.index:
                result[col] = row[col]

        for choice_idx, score in enumerate(scores):
            result[f"score_{CHOICE_LETTERS[choice_idx]}"] = score
            result[f"tokens_{CHOICE_LETTERS[choice_idx]}"] = token_counts[choice_idx]

        rows.append(result)

    return pd.DataFrame(rows)


val_result_df = evaluate_with_loglikelihood(
    val_df,
    desc="Evaluating trained model on val with log-likelihood",
)

val_acc = val_result_df["correct"].mean()
print(f"Validation accuracy (log-likelihood): {val_acc:.4f} ({val_result_df['correct'].sum()}/{len(val_result_df)})")
print("Prediction distribution:")
display(val_result_df["pred_letter"].value_counts().sort_index().rename("count").to_frame())
display(val_result_df.head())

Evaluating trained model on val with log-likelihood:   0%|          | 0/1048 [00:00<?, ?it/s]

Validation accuracy (log-likelihood): 0.7863 (824/1048)
Prediction distribution:


,count
pred_letter,
A,349
B,384
C,244
D,69
E,2


,id,pred,gt,pred_letter,gt_letter,correct,score_margin,question,num_choices,task,...,score_A,tokens_A,score_B,tokens_B,score_C,tokens_C,score_D,tokens_D,score_E,tokens_E
0,val_00671,1,0,B,A,0,3.906174,Why might covering its eggs with its body incr...,3,closed choice,...,-6.492188,1,-0.021561,1,-3.927734,1.0,NaN,NaN,NaN,NaN
1,val_04111,0,1,A,B,0,5.561710,Why might fanning eggs increase the reproducti...,3,closed choice,...,-0.004696,1,-5.566406,1,-9.882812,1.0,NaN,NaN,NaN,NaN
2,val_02022,0,3,A,D,0,5.407310,"Based on clues in the text, how did fossil evi...",4,closed choice,...,-0.010658,1,-5.417969,1,-5.855469,1.0,-5.746094,1.0,NaN,NaN
3,val_01237,1,0,B,A,0,1.187500,What is the probability that a Cepaea snail pr...,5,closed choice,...,-1.883789,1,-0.696289,1,-2.070312,1.0,-2.226562,1.0,-2.179688,1.0
4,val_03458,1,4,B,E,0,0.718750,What is the probability that a Cepaea snail pr...,5,closed choice,...,-1.718750,1,-0.765625,1,-2.734375,1.0,-2.781250,1.0,-1.484375,1.0


In [18]:
# 验证集误差分析

analysis_df = val_result_df.copy()


def summarize_accuracy_by(df, column, min_count=1):
    summary = (
        df.groupby(column, dropna=False)
        .agg(
            n=("correct", "size"),
            accuracy=("correct", "mean"),
            errors=("correct", lambda x: int((1 - x).sum())),
        )
        .reset_index()
    )
    summary = summary[summary["n"] >= min_count]
    return summary.sort_values(["accuracy", "n"], ascending=[True, False])


print("Overall accuracy:", f"{analysis_df['correct'].mean():.4f}")

print("\nAccuracy by number of choices:")
display(summarize_accuracy_by(analysis_df, "num_choices"))

for col in ["subject", "grade", "topic", "category", "skill"]:
    if col in analysis_df.columns:
        print(f"\nWorst groups by {col}:")
        display(summarize_accuracy_by(analysis_df, col, min_count=5).head(10))

print("\nPrediction vs ground truth distribution:")
distribution_df = pd.DataFrame({
    "pred_count": analysis_df["pred_letter"].value_counts().sort_index(),
    "gt_count": analysis_df["gt_letter"].value_counts().sort_index(),
}).fillna(0).astype(int)
display(distribution_df)

print("\nConfusion matrix:")
display(pd.crosstab(
    analysis_df["gt_letter"],
    analysis_df["pred_letter"],
    rownames=["ground_truth"],
    colnames=["prediction"],
))

print("\nHigh-confidence mistakes (small margin means the model was unsure):")
wrong_examples = (
    analysis_df[analysis_df["correct"] == 0]
    .sort_values("score_margin", ascending=False)
    .head(20)
)
display(wrong_examples[[
    "id",
    "gt_letter",
    "pred_letter",
    "score_margin",
    "num_choices",
    "subject",
    "topic",
    "category",
    "skill",
    "question",
]])

Overall accuracy: 0.7863

Accuracy by number of choices:


,num_choices,n,accuracy,errors
3,5,44,0.295455,31
1,3,508,0.769685,117
0,2,244,0.836066,40
2,4,252,0.857143,36



Worst groups by subject:


,subject,n,accuracy,errors
0,language science,28,0.607143,11
1,natural science,777,0.756757,189
2,social science,243,0.901235,24



Worst groups by grade:


...

,pred_count,gt_count
A,349,347
B,384,372
C,244,247
D,69,75
E,2,7



Confusion matrix:


prediction,A,B,C,D,E
ground_truth,,,,,
A,274,40,28,4,1
B,43,305,19,4,1
C,28,27,189,3,0
D,3,9,7,56,0
E,1,3,1,2,0



High-confidence mistakes (small margin means the model was unsure):


,id,gt_letter,pred_letter,score_margin,num_choices,subject,topic,category,skill,question
846,val_00814,D,C,13.999960,4,social science,geography,Maps,Read a map: cardinal directions,Which of these states is farthest west?
737,val_02245,A,B,13.531245,2,natural science,physics,Force and motion,Identify pushes and pulls,Which type of force from the boat causes the w...
213,val_01545,A,C,12.890585,3,natural science,physics,"Velocity, acceleration, and forces",Compare magnitudes of magnetic forces,Think about the magnetic force between the mag...
1017,val_04198,A,B,11.906229,2,natural science,earth-science,Natural resources and human impacts,Evaluate claims about natural resource use: gr...,Select the statement that is supported by the ...
657,val_03949,A,B,10.874959,3,natural science,physics,States of matter,"Identify and sort solids, liquids, and gases","Is iodine a solid, a liquid, or a gas?"
111,val_01657,A,C,10.546800,3,language science,reading-comprehension,Informational texts: level 1,Read passages about animals,"Based on the text, how does a sloth's fur help..."
676,val_03426,B,A,10.531197,4,natural science,biology,Classification,"Identify mammals, birds, fish, reptiles, and a...",Select the mammal below.
1012,val_01177,A,B,10.421789,2,natural science,earth-science,Fossils,Compare ages of fossils in a rock sequence,Which of the following fossils is younger? Sel...
283,val_01486,A,C,9.421751,3,natural science,physics,Magnets,Compare strengths of magnetic forces,Think about the magnetic force between the mag...
279,val_02793,A,C,8.577842,3,natural science,biology,Ecosystems,"Describe populations, communities, and ecosystems",Which of the following best describes an ecosy...


In [ ]:
# 在 test 集上用 multiple-choice log-likelihood 生成 submission file

pred_rows = []

for i in tqdm(range(len(test_df)), desc="Generating submission with log-likelihood"):
    row = test_df.iloc[i]
    pred_idx, scores, token_counts, margin = predict_row_by_loglikelihood(row)

    pred_rows.append({
        "id": row["id"],
        "answer": int(pred_idx),
    })

submission_df = pd.DataFrame(pred_rows)
submission_df.to_csv("submission.csv", index=False)

print(submission_df.head())
print(submission_df.shape)
print("Saved submission.csv with log-likelihood predictions.")

Generating submission with log-likelihood:   0%|          | 0/1008 [00:00<?, ?it/s]

           id  answer
0  test_01750       2
1  test_00128       0
2  test_02891       0
3  test_02425       1
4  test_00930       1
(1008, 2)
Saved submission.csv with log-likelihood predictions.


In [20]:
import shutil

SAVE_SUB_PATH = "/content/drive/MyDrive/final_dl/submission.csv"
shutil.copy("submission.csv", SAVE_SUB_PATH)

print("Saved submission to:", SAVE_SUB_PATH)

Saved submission to: /content/drive/MyDrive/final_dl/submission.csv
